# Phase 1 — Data Pipeline
## Earnings Call Sentiment Engine

This notebook pulls real earnings call transcripts from SEC EDGAR
and stock price data from Yahoo Finance for S&P 500 companies.

## Data Sources
- SEC EDGAR API — official US government filing database
- yfinance — Yahoo Finance stock price data

## Companies Analyzed
Top 10 S&P 500 companies across sectors:
Tech, Finance, Healthcare, Consumer

In [1]:
import requests
import pandas as pd
import yfinance as yf
import json
import time
import os
from datetime import datetime, timedelta
from bs4 import BeautifulSoup


# SEC EDGAR requires a user agent header
HEADERS = {
    'User-Agent': 'Dhaval Vibhakar dhaval.vibhakar@yahoo.com'
}

# Companies to analyze - ticker: company name
COMPANIES = {
    'AAPL': 'Apple Inc',
    'MSFT': 'Microsoft Corporation', 
    'JPM':  'JPMorgan Chase',
    'JNJ':  'Johnson & Johnson',
    'AMZN': 'Amazon',
    'GOOGL': 'Alphabet',
    'BAC':  'Bank of America',
    'PFE':  'Pfizer',
    'V':    'Visa',
    'UNH':  'UnitedHealth Group'
}

print(f"Analyzing {len(COMPANIES)} companies")
print(f"Companies: {list(COMPANIES.keys())}")

/Users/dhaval/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Analyzing 10 companies
Companies: ['AAPL', 'MSFT', 'JPM', 'JNJ', 'AMZN', 'GOOGL', 'BAC', 'PFE', 'V', 'UNH']


In [2]:
# Need to find each company's CIK number on SEC EDGAR
# CIK = Central Index Key, basically SEC's ID for each company
# Found out SEC has a JSON file with all tickers mapped to CIKs - much easier than scraping

def get_cik(ticker):
    # SEC provides a master list of all company tickers and their CIKs
    url = "https://www.sec.gov/files/company_tickers.json"
    response = requests.get(url, headers=HEADERS)
    data = response.json()
    
    # loop through and find our ticker
    for key, company in data.items():
        if company['ticker'].upper() == ticker.upper():
            # CIK needs to be 10 digits with leading zeros
            return str(company['cik_str']).zfill(10)
    
    print(f"Could not find CIK for {ticker}")
    return None

# test with apple first before running all
test = get_cik('AAPL')
print(f"Test - Apple CIK: {test}")

Test - Apple CIK: 0000320193


In [3]:
# get CIKs for all our companies
# sleeping 0.5s between requests so we dont hammer SEC servers
company_ciks = {}

for ticker, name in COMPANIES.items():
    cik = get_cik(ticker)
    company_ciks[ticker] = cik
    print(f"{ticker} ({name}): {cik}")
    time.sleep(0.5)

print(f"\nGot {len(company_ciks)} CIKs")

AAPL (Apple Inc): 0000320193
MSFT (Microsoft Corporation): 0000789019
JPM (JPMorgan Chase): 0000019617
JNJ (Johnson & Johnson): 0000200406
AMZN (Amazon): 0001018724
GOOGL (Alphabet): 0001652044
BAC (Bank of America): 0000070858
PFE (Pfizer): 0000078003
V (Visa): 0001403161
UNH (UnitedHealth Group): 0000731766

Got 10 CIKs


In [10]:
# now we need to find the actual 8-K filings for each company
# 8-K = current report filed when something important happens
# earnings calls are usually filed as 8-K with exhibit 99.1

def get_earnings_filing(cik, company_name):
    """Get the most recent earnings-related 8-K filing"""
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    response = requests.get(url, headers=HEADERS)
    if response.status_code != 200:
        return None
    
    data = response.json()
    filings = data['filings']['recent']
    
    df = pd.DataFrame({
        'form': filings['form'],
        'date': filings['filingDate'],
        'accession': filings['accessionNumber'],
        'description': filings['primaryDocument'],
        'items': filings.get('items', [''] * len(filings['form']))
    })
    
    # filter to 8-K only
    df_8k = df[df['form'] == '8-K'].copy()
    
    # look for earnings related filings
    # Item 2.02 = Results of Operations (earnings releases)
    earnings_keywords = ['2.02', 'result', 'earning', 'quarter', 'revenue']
    
    for _, row in df_8k.iterrows():
        items = str(row.get('items', '')).lower()
        desc = str(row.get('description', '')).lower()
        if any(kw in items or kw in desc for kw in earnings_keywords):
            return row
    
    # if no earnings filing found return most recent
    return df_8k.iloc[0] if not df_8k.empty else None

# test with JPM
print("Testing JPM earnings filing...")
jpm_filing = get_earnings_filing(company_ciks['JPM'], 'JPMorgan')
if jpm_filing is not None:
    print(f"Filing date: {jpm_filing['date']}")
    print(f"Description: {jpm_filing['description']}")
    
    text = get_filing_text(company_ciks['JPM'], jpm_filing['accession'])
    if text:
        print(f"\nText length: {len(text):,}")
        print(f"First 300 chars: {text[:300]}")

Testing JPM earnings filing...
Filing date: 2026-04-14
Description: jpm-20260414.htm

Text length: 22,462
First 300 chars: EX-99.1 2 a1q26erfexhibit991narrative.htm JPMORGAN CHASE & CO. EARNINGS RELEASE - FIRST QUARTER 2026 RESULTS Document Exhibit 99.1 JPMorgan Chase & Co. 270 Park Avenue, New York, NY 10017-2070 NYSE symbol: JPM www.jpmorganchase.com JPMORGANCHASE REPORTS FIRST-QUARTER 2026 NET INCOME OF $16.5 BILLION


In [11]:
# re-pull all companies using earnings-specific function
all_filings = []

for ticker, name in COMPANIES.items():
    print(f"Fetching {name} ({ticker})...")
    cik = company_ciks[ticker]
    
    filing = get_earnings_filing(cik, name)
    
    if filing is None:
        print(f"  No earnings filing found")
        continue
    
    text = get_filing_text(cik, filing['accession'])
    
    if text and len(text) > 500:  # ignore tiny files
        all_filings.append({
            'ticker': ticker,
            'company': name,
            'date': filing['date'],
            'accession': filing['accession'],
            'text': text,
            'text_length': len(text)
        })
        print(f"  Got {len(text):,} chars from {filing['date']}")
    else:
        print(f"  Failed or too short")
    
    time.sleep(1)

df_filings = pd.DataFrame(all_filings)
print(f"\nTotal: {len(df_filings)} companies")
print(df_filings[['ticker', 'company', 'date', 'text_length']])

Fetching Apple Inc (AAPL)...
  Got 10,787 chars from 2026-04-30
Fetching Microsoft Corporation (MSFT)...
  Got 19,717 chars from 2026-04-29
Fetching JPMorgan Chase (JPM)...
  Got 22,462 chars from 2026-04-14
Fetching Johnson & Johnson (JNJ)...
  Got 16,056 chars from 2026-04-14
Fetching Amazon (AMZN)...
  Got 36,092 chars from 2026-04-29
Fetching Alphabet (GOOGL)...
  Got 22,168 chars from 2026-04-29
Fetching Bank of America (BAC)...
  Got 60,607 chars from 2026-04-15
Fetching Pfizer (PFE)...
  Got 69,755 chars from 2026-05-05
Fetching Visa (V)...
  Got 29,418 chars from 2026-04-28
Fetching UnitedHealth Group (UNH)...
  Got 44,900 chars from 2026-04-21

Total: 10 companies
  ticker                company        date  text_length
0   AAPL              Apple Inc  2026-04-30        10787
1   MSFT  Microsoft Corporation  2026-04-29        19717
2    JPM         JPMorgan Chase  2026-04-14        22462
3    JNJ      Johnson & Johnson  2026-04-14        16056
4   AMZN                 Amazon  

In [12]:
# check JPM text quality
jpm_text = df_filings[df_filings['ticker'] == 'JPM']['text'].values[0]
print(f"JPM text length: {len(jpm_text)}")
print(f"\nFirst 500 chars:")
print(jpm_text[:500])

JPM text length: 22462

First 500 chars:
EX-99.1 2 a1q26erfexhibit991narrative.htm JPMORGAN CHASE & CO. EARNINGS RELEASE - FIRST QUARTER 2026 RESULTS Document Exhibit 99.1 JPMorgan Chase & Co. 270 Park Avenue, New York, NY 10017-2070 NYSE symbol: JPM www.jpmorganchase.com JPMORGANCHASE REPORTS FIRST-QUARTER 2026 NET INCOME OF $16.5 BILLION ( $5.94 PER SHARE) FIRST-QUARTER 2026 RESULTS 1 ROE 19% ROTCE 2 23% CET1 Capital Ratios 3 Std. 14.3% | Adv. 14.1% Total Loss-Absorbing Capacity 3 $572B Std. RWA 3 $2.0T Cash and marketable securities


In [13]:
# save filings to CSV - save everything except the full text
# text is too large for CSV, save separately

import os
os.makedirs('data', exist_ok=True)

# save metadata
df_meta = df_filings[['ticker', 'company', 'date', 'accession', 'text_length']]
df_meta.to_csv('data/filings_metadata.csv', index=False)
print("Saved metadata to data/filings_metadata.csv")

# save each company's text as separate file
os.makedirs('data/texts', exist_ok=True)
for _, row in df_filings.iterrows():
    filename = f"data/texts/{row['ticker']}_earnings.txt"
    with open(filename, 'w') as f:
        f.write(row['text'])
    print(f"Saved {row['ticker']} text to {filename}")

print("\nAll data saved successfully")

Saved metadata to data/filings_metadata.csv
Saved AAPL text to data/texts/AAPL_earnings.txt
Saved MSFT text to data/texts/MSFT_earnings.txt
Saved JPM text to data/texts/JPM_earnings.txt
Saved JNJ text to data/texts/JNJ_earnings.txt
Saved AMZN text to data/texts/AMZN_earnings.txt
Saved GOOGL text to data/texts/GOOGL_earnings.txt
Saved BAC text to data/texts/BAC_earnings.txt
Saved PFE text to data/texts/PFE_earnings.txt
Saved V text to data/texts/V_earnings.txt
Saved UNH text to data/texts/UNH_earnings.txt

All data saved successfully
